# Conformal predictive systems for time series

This notebook covers discrete demand and continuous measurements with Nixtla-compatible panel forecasters. Both CPS variants calibrate series- and horizon-specific residual distributions and return a self-contained, panel-aligned forecast.

In [23]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from IPython.display import display
from mlforecast import MLForecast
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from tinyconformal.series import (
    ContinuousTimeSeriesConformalPredictiveSystem,
    DiscreteTimeSeriesConformalPredictiveSystem,
)
from tinyconformal.utils import FirstStageEvaluator, NewsvendorSolver

pd.set_option("display.max_columns", 20)

## 1. Discrete demand

The target is a non-negative integer count. The last seven days are held out so evaluation uses observations that were not used during fitting.

In [24]:
def make_count_panel(n_periods=140, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["store_A", "store_B"]):
        mean = 4.0 + offset + np.linspace(0, 1.5, n_periods) + 1.5 * (dates.dayofweek >= 5)
        frames.append(pd.DataFrame({
            "unique_id": unique_id, "ds": dates, "y": rng.poisson(mean)
        }))
    return pd.concat(frames, ignore_index=True)

horizon = 7
count_data = make_count_panel()
count_train = count_data.groupby("unique_id", group_keys=False).head(-horizon)
count_test = count_data.groupby("unique_id", group_keys=False).tail(horizon)
count_data.head()

,unique_id,ds,y
0,store_A,2025-01-01,6
1,store_A,2025-01-02,3
2,store_A,2025-01-03,6
3,store_A,2025-01-04,9
4,store_A,2025-01-05,7


In [25]:
count_learner = MLForecast(
    models={"LinearRegression": LinearRegression()},
    freq="D", lags=[1, 7, 14], date_features=["dayofweek"],
)
count_cps = DiscreteTimeSeriesConformalPredictiveSystem(
    learner=count_learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=100, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    horizon=horizon, n_windows=5, minimum=0,
).fit(count_train, static_features=[], n_jobs=1)
count_forecast = count_cps.predict_distribution(h=horizon)
count_forecast.to_frame().head()

,unique_id,ds,LinearRegression
0,store_A,2025-05-14,4.378222
1,store_A,2025-05-15,4.524704
2,store_A,2025-05-16,4.935391
3,store_A,2025-05-17,5.504575
4,store_A,2025-05-18,6.329747


### Discrete PPF, CDF, SF, and PMF

Each method returns a DataFrame on the same `unique_id`/`ds` grid. For stock level `N`, `cdf(N)` is the service probability `P(Y<=N)` and `sf(N)` is the exceedance risk `P(Y>N)`.

In [26]:
display(count_forecast.ppf([0.50, 0.90, 0.95]).head())
inventory_level = 8
service = count_forecast.cdf(inventory_level)
risk = count_forecast.sf(inventory_level)
probabilities = service.join(risk[[f"P(Y>{inventory_level})"]])
probabilities["probability_check"] = (
    probabilities[f"P(Y<={inventory_level})"] + probabilities[f"P(Y>{inventory_level})"]
)
display(probabilities.head())
count_forecast.pmf([0, 5, 8]).head()

,unique_id,ds,LinearRegression,Q(0.5),Q(0.9),Q(0.95)
0,store_A,2025-05-14,4.378222,3,7,7
1,store_A,2025-05-15,4.524704,2,7,7
2,store_A,2025-05-16,4.935391,2,14,14
3,store_A,2025-05-17,5.504575,6,10,10
4,store_A,2025-05-18,6.329747,8,14,14


,unique_id,ds,LinearRegression,P(Y<=8),P(Y>8),probability_check
0,store_A,2025-05-14,4.378222,1.000000,0.000000,1.0
1,store_A,2025-05-15,4.524704,1.000000,0.000000,1.0
2,store_A,2025-05-16,4.935391,0.666667,0.333333,1.0
3,store_A,2025-05-17,5.504575,0.666667,0.333333,1.0
4,store_A,2025-05-18,6.329747,0.500000,0.500000,1.0


,unique_id,ds,LinearRegression,P(Y=0),P(Y=5),P(Y=8)
0,store_A,2025-05-14,4.378222,0.166667,0.000000,0.000000
1,store_A,2025-05-15,4.524704,0.166667,0.000000,0.000000
2,store_A,2025-05-16,4.935391,0.000000,0.000000,0.000000
3,store_A,2025-05-17,5.504575,0.000000,0.166667,0.166667
4,store_A,2025-05-18,6.329747,0.166667,0.000000,0.166667


### Held-out evaluation and first-stage diagnostics

The predictive forecast evaluates complete central intervals. The first-stage diagnostics separately summarize point forecasts from a rolling backtest.

In [34]:
count_observed = count_test.sort_values(["unique_id", "ds"])["y"].to_numpy()
display(count_forecast.evaluate(count_observed, coverages=[0.80, 0.90, 0.95]))

count_backtest = count_learner.cross_validation(
    count_train, n_windows=3, h=horizon, static_features=[]
)
display(FirstStageEvaluator.evaluate(
    count_backtest, prediction_col="LinearRegression", time_col="ds", id_col="unique_id"
) )
FirstStageEvaluator.calibration_table(
    count_backtest, prediction_col="LinearRegression", n_bins=5
)

,coverage,coverage_rate,interval_width_mean,mwis
0,0.80,0.786,8.0,11.571
1,0.90,0.786,8.0,15.143
2,0.95,0.786,8.0,22.286


,wape,pbias,score,forecast_instability,false_demand_on_zero_days_avg_pred,peak_demand_deviation
0,0.4188,0.0421,0.4609,0.1394,0.0,0.0421


,calibration_bin,count,mean_prediction,mean_observed,mean_residual
0,"(4.399, 5.2]",9,4.779201,4.333333,-0.445867
1,"(5.2, 5.768]",8,5.456273,6.125000,0.668727
2,"(5.768, 6.559]",8,6.038697,5.125000,-0.913697
3,"(6.559, 7.015]",8,6.794840,6.375000,-0.419840
4,"(7.015, 7.912]",9,7.330935,7.222222,-0.108713


### Optimize discrete inventory

A time-series CPS forecast can be passed directly to the solver. Its panel and underlying distribution are already aligned.

In [28]:
count_plan = NewsvendorSolver.optimize_distribution(
    count_forecast, underage_cost=9.0, overage_cost=1.0
)
display(count_plan.head())
NewsvendorSolver.marginal_benefit_distribution(
    count_forecast, underage_cost=9.0, overage_cost=1.0, units=range(0, 11, 2)
).head()

,unique_id,ds,LinearRegression,critical_ratio,y_optimal
0,store_A,2025-05-14,4.378222,0.9,7.0
1,store_A,2025-05-15,4.524704,0.9,7.0
2,store_A,2025-05-16,4.935391,0.9,14.0
3,store_A,2025-05-17,5.504575,0.9,10.0
4,store_A,2025-05-18,6.329747,0.9,14.0


,unique_id,ds,LinearRegression,MB(k=0),MB(k=2),MB(k=4),MB(k=6),MB(k=8),MB(k=10)
0,store_A,2025-05-14,4.378222,9.0,7.333333,4.000000,2.333333,-1.000000,-1.000000
1,store_A,2025-05-15,4.524704,9.0,5.666667,2.333333,2.333333,-1.000000,-1.000000
2,store_A,2025-05-16,4.935391,9.0,7.333333,4.000000,4.000000,2.333333,2.333333
3,store_A,2025-05-17,5.504575,9.0,7.333333,7.333333,5.666667,4.000000,2.333333
4,store_A,2025-05-18,6.329747,9.0,7.333333,7.333333,7.333333,5.666667,2.333333


## 2. Continuous measurements

Continuous CPS uses the same panel workflow, but retains real-valued support and therefore does not expose a PMF.

In [29]:
def make_continuous_panel(n_periods=140, seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["region_A", "region_B"]):
        t = np.arange(n_periods)
        y = (10 + 2 * offset + 0.02 * t + 1.5 * np.sin(2 * np.pi * t / 7)
             + rng.normal(0, 0.8 + 0.2 * offset, n_periods))
        frames.append(pd.DataFrame({"unique_id": unique_id, "ds": dates, "y": y}))
    return pd.concat(frames, ignore_index=True)

continuous_data = make_continuous_panel()
continuous_train = continuous_data.groupby("unique_id", group_keys=False).head(-horizon)
continuous_test = continuous_data.groupby("unique_id", group_keys=False).tail(horizon)

In [30]:
continuous_learner = MLForecast(
    models={"LinearRegression": LinearRegression()},
    freq="D", lags=[1, 7, 14], date_features=["dayofweek"],
)
continuous_cps = ContinuousTimeSeriesConformalPredictiveSystem(
    learner=continuous_learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=100, min_samples_leaf=3, random_state=43, n_jobs=-1
    ),
    horizon=horizon, n_windows=5,
).fit(continuous_train, static_features=[], n_jobs=1)
continuous_forecast = continuous_cps.predict_distribution(h=horizon)

### Continuous PPF, CDF, SF, intervals, and decisions

The distribution interface is the same as in the discrete case, except that quantiles and optimal quantities remain continuous.

In [31]:
display(continuous_forecast.ppf([0.05, 0.50, 0.95]).head())
display(continuous_forecast.cdf([10, 15]).head())
display(continuous_forecast.sf([10, 15]).head())
display(continuous_forecast.interval(coverage=0.90).head())

continuous_observed = continuous_test.sort_values(["unique_id", "ds"])["y"].to_numpy()
display(continuous_forecast.evaluate(
    continuous_observed, coverages=[0.80, 0.90, 0.95]
) )
NewsvendorSolver.optimize_distribution(
    continuous_forecast, underage_cost=6.0, overage_cost=2.0
).head()

,unique_id,ds,LinearRegression,Q(0.05),Q(0.5),Q(0.95)
0,region_A,2025-05-14,11.763821,9.990155,11.744869,12.478141
1,region_A,2025-05-15,13.740267,12.573894,14.582093,15.203459
2,region_A,2025-05-16,13.172945,12.077639,12.861564,13.710393
3,region_A,2025-05-17,13.485322,12.761572,13.623544,14.615739
4,region_A,2025-05-18,11.480983,9.009745,10.962213,11.543327


,unique_id,ds,LinearRegression,P(Y<=10),P(Y<=15)
0,region_A,2025-05-14,11.763821,0.166667,1.000000
1,region_A,2025-05-15,13.740267,0.000000,0.666667
2,region_A,2025-05-16,13.172945,0.000000,1.000000
3,region_A,2025-05-17,13.485322,0.000000,1.000000
4,region_A,2025-05-18,11.480983,0.166667,1.000000


,unique_id,ds,LinearRegression,P(Y>10),P(Y>15)
0,region_A,2025-05-14,11.763821,0.833333,0.000000
1,region_A,2025-05-15,13.740267,1.000000,0.333333
2,region_A,2025-05-16,13.172945,1.000000,0.000000
3,region_A,2025-05-17,13.485322,1.000000,0.000000
4,region_A,2025-05-18,11.480983,0.833333,0.000000


,unique_id,ds,LinearRegression,Q(0.05),Q(0.95)
0,region_A,2025-05-14,11.763821,9.990155,12.478141
1,region_A,2025-05-15,13.740267,12.573894,15.203459
2,region_A,2025-05-16,13.172945,12.077639,13.710393
3,region_A,2025-05-17,13.485322,12.761572,14.615739
4,region_A,2025-05-18,11.480983,9.009745,11.543327


,coverage,coverage_rate,interval_width_mean,mwis
0,0.80,0.643,3.288,5.569
1,0.90,0.643,3.288,7.850
2,0.95,0.643,3.288,12.411


,unique_id,ds,LinearRegression,critical_ratio,y_optimal
0,region_A,2025-05-14,11.763821,0.75,12.478141
1,region_A,2025-05-15,13.740267,0.75,15.203459
2,region_A,2025-05-16,13.172945,0.75,13.710393
3,region_A,2025-05-17,13.485322,0.75,14.615739
4,region_A,2025-05-18,11.480983,0.75,11.543327


## Support comparison

| Target | CDF | SF | PPF | PMF | Newsvendor output |
|---|---:|---:|---:|---:|---|
| Non-negative integer counts | Yes | Yes | Yes | Yes | Integer |
| Continuous values | Yes | Yes | Yes | No | Continuous |

Unlike the tabular cross-conformal CPS, time-series calibration is performed separately by series and forecast horizon using sequential rolling-origin windows.